# Lab 2 — Partner A: what drives the model **overall** (global SHAP)

Module 2 is about *explaining* models, not just fitting them. You own the **global** question:
across all 743 hours, **which features move the prediction the most, and in which direction?**

Your night:
1. Run the setup + SHAP setup (given)
2. Look at the model's **built-in importances** (given) - the quick-but-shallow answer
3. **Write one line** to make the **SHAP beeswarm** - the honest answer (your only coding task)
4. Ship the two PNGs + this notebook through a pull request on branch `dev-global`

> Type the one SHAP line yourself. Module 3 is where we lean on AI tools; today is about *seeing* it.

In [ ]:
# --- setup: install SHAP, load the data, fit the model (just run this) ---
!pip install shap -q

import pandas as pd, numpy as np, matplotlib.pyplot as plt, shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

url = "https://raw.githubusercontent.com/drdave-teaching/opim5512-lab2-template/main/data/energy_model_data.csv"
df = pd.read_csv(url, parse_dates=["hour"])

FEATURES = ["temp_f", "hour_of_day", "dewpoint_f", "humidity_pct", "wind_kt", "weekend"]
X, y = df[FEATURES], df["load_mw"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print(f"model R2 (test): {r2_score(yte, model.predict(Xte)):.2f}   |   typical miss: {mean_absolute_error(yte, model.predict(Xte)):,.0f} MW")
X.head()

### How the model was built (read this — don't code it tonight)

We reused the joined weather + demand data **you built in Lab 1** — one row per hour — and trained a
small **random forest** to predict New England electricity demand from six features. It's accurate
(R^2 around 0.9), which makes it worth asking the real question of Module 2: **how does it decide?**

| feature | meaning | units |
|---|---|---|
| `temp_f` | air temperature | deg F |
| `hour_of_day` | 0-23, the hour the reading begins | hour |
| `dewpoint_f` | dew point (muggy-ness) | deg F |
| `humidity_pct` | relative humidity | % |
| `wind_kt` | wind speed | knots |
| `weekend` | 1 on Sat/Sun, else 0 | 0/1 |

**SHAP** answers "how did the model decide" by giving every feature, for every prediction, a number
in **MW**: how much it pushed that prediction **up (+)** or **down (-)** from the average. Add them
all up and you get the model's prediction. That's the whole idea.

In [ ]:
# --- SHAP setup (just run this): explain every prediction the model makes ---
explainer = shap.TreeExplainer(model)
shap_values = explainer(X)          # one row of SHAP values per hour, one column per feature
print("SHAP ready:", shap_values.shape, "(hours x features)")

## 1. The quick answer (given): the model's built-in importances

Every tree model can rank its features. Run this - it saves `importances_builtin.png`.

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
ax = imp.plot.barh(figsize=(8,4), title="Model's built-in feature importances (magnitude only)")
ax.set_xlabel("importance")
ax.get_figure().savefig("importances_builtin.png", dpi=150, bbox_inches="tight")
print("saved importances_builtin.png")

## 2. Your turn: the SHAP beeswarm (one line)

Built-in importances tell you a feature *matters* - but not **which way** it pushes, or for whom.
The SHAP **beeswarm** does: each dot is one hour, colored by the feature's value, placed by how many
MW it added or removed. Write it:

```
shap.plots.beeswarm(shap_values, show=False)
plt.gcf().savefig("shap_global.png", dpi=150, bbox_inches="tight"); plt.close()
```

Then **look at it:** which feature is #1? Do high (red) values push demand up or down? Does SHAP agree
with the built-in ranking above - or reorder it?

In [ ]:
# TODO: your SHAP beeswarm. Save it as shap_global.png (exact name - the report links to it).



### Download both PNGs to your laptop

In [ ]:
import os
from google.colab import files
for p in ['importances_builtin.png', 'shap_global.png']:
    if os.path.exists(p):
        files.download(p)
    else:
        print(f"{p} not found yet - run the cell that makes it, then re-run this one.")

## Ship it (this is the git half of the lab)

1. **This notebook -> GitHub:** **File -> Save** -> your repo, branch **`dev-global`**, keep the path
   `notebooks/Lab2_A_Global_SHAP.ipynb`, real commit message. (Plain **Save** commits to GitHub because you opened it
   *from* GitHub. Ctrl+S only autosaves to Drive.)
2. **The two PNGs -> your repo:** the download cell put them in your Downloads folder. GitHub Desktop
   -> **Repository -> Show in Explorer** -> drag `importances_builtin.png` and `shap_global.png` into **`images/`**
   (rename any `(1)` copy back first).
3. GitHub Desktop: on branch **`dev-global`** -> **Commit** (real message) -> **Push** ->
   github.com **Compare & pull request** -> ask your partner to review.